In [1]:
# Checking the data
DATA_DIR = "../test_data"
PLOTS_DIR = "../results/plots"
RESULTS_DIR = "../results"

In [2]:
# Imports
import os, glob, time
import numpy as np
import pandas as pd
import librosa, librosa.display
import pretty_midi
import matplotlib.pyplot as plt
import torch, torch.nn as nn

In [3]:
# Finding the wav + midi pairs in the data directory
os.makedirs(PLOTS_DIR, exist_ok=True)
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"

def find_pairs(data_dir):
    midi_by_stem = {}
    for path in glob.glob(os.path.join(data_dir, "**", "*"), recursive=True):
        if os.path.isfile(path) and os.path.splitext(path)[1].lower() in (".mid", ".midi"):
            stem = os.path.normcase(os.path.splitext(path)[0])
            midi_by_stem[stem] = path

    pairs = []
    for wav in sorted(glob.glob(os.path.join(data_dir, "**", "*.wav"), recursive=True)):
        stem = os.path.normcase(os.path.splitext(wav)[0])
        if stem in midi_by_stem:
            pairs.append((wav, midi_by_stem[stem]))
    return pairs

pairs = find_pairs(DATA_DIR)
print(f"Found {len(pairs)} WAV + MIDI pairs")
if not pairs:
    print(f"Warning: no matching files found under {os.path.abspath(DATA_DIR)}. Check DATA_DIR and the file locations.")

SECONDS = 120   # only use the first 2 minutes of each file, so it runs fast
rows = []

Found 10 WAV + MIDI pairs


In [ ]:
# Model Parameters
SR, N_MELS, HOP, SEG_SECONDS = 22050, 128, 512, 5
MAX_SECONDS = 180
EPOCHS, BATCH, SEED = 8, 16, 42   # SEED also fixes the train/val/test split
SEEDS = [42, 7, 123]              # every model is trained once per seed; results are reported as mean ± std
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FRAMES = int(SEG_SECONDS * SR / HOP)   # frames per 5 s segment
FRAME_TIMES = np.arange(FRAMES) * HOP / SR
print("Training on:", DEVICE)

In [5]:
# Turning MIDI notes into per-frame targets

# dynamics: mean velocity of the notes sounding at each frame, scaled to 0-1
def dynamics_curve(notes, times):
    st = np.array([n.start for n in notes]); en = np.array([n.end for n in notes])
    vl = np.array([n.velocity for n in notes], dtype=float)
    out = np.zeros(len(times))
    for i, t in enumerate(times):
        on = (st <= t) & (en > t)
        if on.any(): out[i] = vl[on].mean()
    return out / 127.0

# note start times, with chords merged into one onset
def onset_times(notes, gap=0.05):
    s = np.sort([n.start for n in notes]); keep = [s[0]]
    for t in s[1:]:
        if t - keep[-1] > gap: keep.append(t)   # a chord counts as one onset
    return np.array(keep)

# tempo: local onset rate vs the piece's usual rate, as log2 (-1 = half speed, +1 = double)
def tempo_curve(notes, times, window=3.0):
    on = onset_times(notes)
    rate = np.array([((on >= t - window/2) & (on < t + window/2)).sum() / window for t in times])
    usual = np.median(rate[rate > 0]) if np.any(rate > 0) else 1.0
    return np.log2(np.clip(rate / usual, 0.5, 2.0))

In [6]:
# Building mel inputs + targets for every piece (saved to disk after the first run)
CACHE = os.path.join(RESULTS_DIR, f"features_{N_MELS}_{HOP}_{SEG_SECONDS}s.npz")

# one piece -> 5 s segments of mel frames (X) and dynamics/tempo targets (Y)
def build_piece(w, m):
    y, _ = librosa.load(w, sr=SR, mono=True, duration=MAX_SECONDS)
    mel = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, hop_length=HOP), ref=1.0)
    notes = [n for i in pretty_midi.PrettyMIDI(m).instruments for n in i.notes]
    t = np.arange(mel.shape[1]) * HOP / SR
    dyn, tmp = dynamics_curve(notes, t), tempo_curve(notes, t)
    X, Y = [], []
    for k in range(mel.shape[1] // FRAMES):
        sl = slice(k*FRAMES, (k+1)*FRAMES)
        X.append(mel[:, sl]); Y.append(np.stack([dyn[sl], tmp[sl]], axis=1))
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

if os.path.exists(CACHE):
    d = np.load(CACHE, allow_pickle=True)
    X_all, Y_all, piece_id, names = d["X"], d["Y"], d["piece"], list(d["names"])
else:
    Xs, Ys, ids, names = [], [], [], []
    for i, (w, m) in enumerate(pairs):
        X, Y = build_piece(w, m)
        Xs.append(X); Ys.append(Y); ids += [i]*len(X); names.append(os.path.basename(w))
        print(f"{i+1}/{len(pairs)} done: {len(X)} segments")
    X_all, Y_all, piece_id = np.concatenate(Xs), np.concatenate(Ys), np.array(ids)
    np.savez(CACHE, X=X_all, Y=Y_all, piece=piece_id, names=np.array(names))
print("Inputs:", X_all.shape, " Targets:", Y_all.shape)

Inputs: (325, 128, 215)  Targets: (325, 215, 2)


In [7]:
# Splitting by piece, so no piece appears in two splits
order = np.random.default_rng(SEED).permutation(len(pairs))
n_test = n_val = max(1, round(0.2 * len(pairs)))
test_p, val_p, train_p = order[:n_test], order[n_test:n_test+n_val], order[n_test+n_val:]
split = {"train": np.isin(piece_id, train_p), "val": np.isin(piece_id, val_p), "test": np.isin(piece_id, test_p)}
for s, m in split.items(): print(f"{s}: {m.sum()} segments")

# normalising each mel band with train statistics only (no leakage)
mu = X_all[split["train"]].mean(axis=(0, 2), keepdims=True)
sd = X_all[split["train"]].std(axis=(0, 2), keepdims=True) + 1e-6
Xn = (X_all - mu) / sd
def tensors(s): return torch.tensor(Xn[split[s]]).unsqueeze(1), torch.tensor(Y_all[split[s]])
Xtr, Ytr = tensors("train"); Xva, Yva = tensors("val"); Xte, Yte = tensors("test")

train: 197 segments
val: 56 segments
test: 72 segments


In [ ]:
# Metrics for both targets, plus a baseline that always predicts the train mean
#   MAE / RMSE: average error size (dynamics in velocity units 0-127, tempo in log2 units)
#   R2:  share of the variation explained (1 = perfect, 0 = no better than predicting the mean)
#   r:   do predictions rise and fall together with the ground truth (Pearson correlation)
#   CCC: like r, but also penalises predictions that are shifted or squashed (concordance correlation)
def _r(a, b): return float(np.corrcoef(a, b)[0, 1]) if a.std() > 1e-6 and b.std() > 1e-6 else np.nan
def _ccc(a, b): return float(2 * np.cov(a, b, bias=True)[0, 1] / (a.var() + b.var() + (a.mean() - b.mean())**2 + 1e-12))

def scores(pred, true):
    p, t = pred.numpy(), true.numpy()
    out = {}
    for k, name, scale in ((0, "Dyn", 127), (1, "Tempo", 1)):
        a, b = p[..., k].ravel(), t[..., k].ravel(); e = a - b
        out[f"{name} MAE"] = float(np.abs(e).mean() * scale)
        out[f"{name} RMSE"] = float(np.sqrt((e**2).mean()) * scale)
        out[f"{name} R2"] = float(1 - (e**2).sum() / ((b - b.mean())**2).sum())
        out[f"{name} r"] = _r(a, b)
        out[f"{name} CCC"] = _ccc(a, b)
    # dynamics error only where notes are sounding (silent frames have a target of 0)
    on = t[..., 0] > 0
    out["Dyn MAE (notes only)"] = float(np.abs(p[..., 0][on] - t[..., 0][on]).mean() * 127)
    # does the dynamics curve rise and fall at the right times inside each 5 s segment?
    seg_r = np.array([_r(p[i, :, 0], t[i, :, 0]) for i in range(len(t))])
    out["Dyn shape r (per segment)"] = float(np.nanmean(seg_r)) if np.isfinite(seg_r).any() else np.nan
    # where the ground truth clearly speeds up or slows down, is the predicted direction right?
    clear = np.abs(t[..., 1]) >= 0.1
    out["Tempo direction acc"] = float((np.sign(p[..., 1][clear]) == np.sign(t[..., 1][clear])).mean())
    return out

runs = [{"Model": "Baseline (mean)", "Seed": SEED, "Params": 0, "Train time (s)": 0.0, "Infer (ms/seg)": 0.0,
         **scores(Ytr.mean(dim=(0, 1)).expand_as(Yte).clone(), Yte)}]
pd.DataFrame(runs).round(3).T

In [ ]:
# The three models; all share the same CNN front end

# CNN: shrinks the mel axis, keeps every time frame
class CNNFrontEnd(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d((2, 1)))
        self.net = nn.Sequential(block(1, 32), block(32, 64), block(64, 64))
        self.out_dim = 64 * (N_MELS // 8)
    def forward(self, x): return self.net(x).flatten(1, 2).transpose(1, 2)

# CNN only: each frame predicted on its own
class CNNOnly(nn.Module):
    def __init__(self):
        super().__init__(); self.cnn = CNNFrontEnd()
        self.head = nn.Sequential(nn.Linear(self.cnn.out_dim, 128), nn.ReLU(), nn.Linear(128, 2))
    def forward(self, x): return self.head(self.cnn(x))

# CRNN: a bidirectional GRU adds context across time
class CRNN(nn.Module):
    def __init__(self):
        super().__init__(); self.cnn = CNNFrontEnd()
        self.proj = nn.Linear(self.cnn.out_dim, 128)
        self.rnn = nn.GRU(128, 64, batch_first=True, bidirectional=True)
        self.head = nn.Linear(128, 2)
    def forward(self, x):
        h, _ = self.rnn(torch.relu(self.proj(self.cnn(x)))); return self.head(h)

# CNN + Transformer: self-attention adds context across time
class CNNTransformer(nn.Module):
    def __init__(self):
        super().__init__(); self.cnn = CNNFrontEnd()
        self.proj = nn.Linear(self.cnn.out_dim, 128)
        self.pos = nn.Parameter(torch.zeros(1, FRAMES, 128))
        layer = nn.TransformerEncoderLayer(128, 4, 256, dropout=0.1, batch_first=True)
        self.tf = nn.TransformerEncoder(layer, 2); self.head = nn.Linear(128, 2)
    def forward(self, x): return self.head(self.tf(self.proj(self.cnn(x)) + self.pos[:, :x.shape[-1]]))

# quick shape and size check
# Transformer only: no CNN, each mel frame goes straight into self-attention
class TransformerOnly(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(N_MELS, 128)
        self.pos = nn.Parameter(torch.zeros(1, FRAMES, 128))
        layer = nn.TransformerEncoderLayer(128, 4, 256, dropout=0.1, batch_first=True)
        self.tf = nn.TransformerEncoder(layer, 2); self.head = nn.Linear(128, 2)
    def forward(self, x): return self.head(self.tf(self.proj(x.squeeze(1).transpose(1, 2)) + self.pos[:, :x.shape[-1]]))

dummy = torch.zeros(2, 1, N_MELS, FRAMES)
for M in (CNNOnly, CRNN, CNNTransformer, TransformerOnly):
    m = M(); print(M.__name__, tuple(m(dummy).shape), f"{sum(p.numel() for p in m.parameters()):,} params")

In [ ]:
# Training loop: keeps the epoch with the best validation loss
MODEL_DIR = os.path.join(RESULTS_DIR, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
RETRAIN = False   # set True after changing the data, the models or the training settings

def ckpt_path(name, seed):
    slug = "".join(ch for ch in name.replace("+", "_") if ch.isalnum() or ch in "_-")
    return os.path.join(MODEL_DIR, f"{slug}_{EPOCHS}ep_seed{seed}_v2.pt")

def predict_test(model):
    # test-set predictions plus inference speed (ms per 5 s segment)
    model.eval()
    with torch.no_grad():
        model(Xte[:2].to(DEVICE)); t0 = time.time()
        pred = model(Xte.to(DEVICE)).cpu()
    return pred, (time.time() - t0) / len(Xte) * 1000

def train_model(make_model, name, seed, warmup_epochs=0):
    torch.manual_seed(seed); model = make_model().to(DEVICE)
    ckpt = ckpt_path(name, seed)
    if os.path.exists(ckpt) and not RETRAIN:
        # reuse a saved run so restarting the kernel doesn't mean retraining
        saved = torch.load(ckpt, weights_only=True)
        model.load_state_dict(saved["state"]); hist, t = saved["hist"], saved["time"]
        print(f"{name} (seed {seed}): loaded saved run")
    else:
        opt = torch.optim.Adam(model.parameters(), lr=1e-3); loss_fn = nn.MSELoss()
        warm = warmup_epochs * -(-len(Xtr) // BATCH)   # warm-up length in batches
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) if warm else 1.0)
        hist, best, best_state = {"train": [], "val": []}, float("inf"), None
        t0 = time.time()
        for ep in range(EPOCHS):
            model.train(); perm = torch.randperm(len(Xtr)); total = 0
            for i in range(0, len(Xtr), BATCH):
                idx = perm[i:i+BATCH]; xb, yb = Xtr[idx].to(DEVICE), Ytr[idx].to(DEVICE)
                opt.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); opt.step(); sched.step()
                total += loss.item() * len(idx)
            model.eval()
            with torch.no_grad(): val = loss_fn(model(Xva.to(DEVICE)), Yva.to(DEVICE)).item()
            hist["train"].append(total/len(Xtr)); hist["val"].append(val)
            if val < best: best, best_state = val, {k: v.clone() for k, v in model.state_dict().items()}
            print(f"{name} (seed {seed}) epoch {ep+1}: train {hist['train'][-1]:.4f}  val {val:.4f}  ({time.time()-t0:.0f} s)")
        t = round(time.time() - t0, 1)
        model.load_state_dict(best_state)
        torch.save({"state": {k: v.cpu() for k, v in best_state.items()}, "hist": hist, "time": t}, ckpt)
    pred, ms = predict_test(model)
    res = {"Model": name, "Seed": seed, "Params": sum(p.numel() for p in model.parameters()),
           "Train time (s)": t, "Infer (ms/seg)": round(ms, 2), **scores(pred, Yte)}
    return res, hist, pred

In [ ]:
# Training (or loading) every model once per seed
MODELS = {   # name: (model class, warm-up epochs)
    "CNN only": (CNNOnly, 0),
    "CRNN": (CRNN, 0),
    "CNN + Transformer": (CNNTransformer, 0),
    "CNN + Transformer (warm-up)": (CNNTransformer, 2),   # Transformers often train more stably with a learning-rate warm-up
    "Transformer only": (TransformerOnly, 0),             # does the Transformer need the CNN front end?
}
histories, preds = {n: [] for n in MODELS}, {}
for name, (M, warm) in MODELS.items():
    for seed in SEEDS:
        res, hist, pred = train_model(M, name, seed, warm)
        runs.append(res); histories[name].append(hist)
        if seed == SEEDS[0]: preds[name] = pred

In [ ]:
# Comparison table: mean ± std over the seeds, with the best model for each metric
BETTER = {"Dyn MAE": "low", "Dyn RMSE": "low", "Dyn R2": "high", "Dyn r": "high", "Dyn CCC": "high",
          "Dyn MAE (notes only)": "low", "Dyn shape r (per segment)": "high",
          "Tempo MAE": "low", "Tempo RMSE": "low", "Tempo R2": "high", "Tempo r": "high", "Tempo CCC": "high",
          "Tempo direction acc": "high", "Params": "low", "Train time (s)": "low", "Infer (ms/seg)": "low"}
runs_df = pd.DataFrame(runs)
runs_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_runs.csv"), index=False)
mean = runs_df.groupby("Model", sort=False)[list(BETTER)].mean()
std = runs_df.groupby("Model", sort=False)[list(BETTER)].std().fillna(0)
pd.concat({"mean": mean, "std": std}, axis=1).round(4).to_csv(os.path.join(RESULTS_DIR, "model_comparison_summary.csv"))

def cell(m, s, col):
    if np.isnan(m): return "-"
    if col == "Params": return f"{int(m):,}"
    return f"{m:.3f} ± {s:.3f}" if s else f"{m:.3f}"

trained = mean.drop(index="Baseline (mean)")
table = pd.DataFrame({col: [cell(mean.loc[i, col], std.loc[i, col], col) for i in mean.index] for col in BETTER},
                     index=mean.index).T
table["Better"] = [BETTER[c] for c in table.index]
table["Best model"] = [(trained[c].idxmin() if BETTER[c] == "low" else trained[c].idxmax()) for c in table.index]
table

In [ ]:
# Train and validation loss curves (mean over seeds)
colors = {"CNN only": BLUE, "CRNN": ORANGE, "CNN + Transformer": AQUA,
          "CNN + Transformer (warm-up)": YELLOW, "Transformer only": MAGENTA}
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for ax, part in zip(axes, ("train", "val")):
    for n, hs in histories.items():
        curve = np.mean([h[part] for h in hs], axis=0)
        ax.plot(range(1, len(curve)+1), curve, color=colors[n], lw=2, label=n)
    ax.set_title(f"{part} loss (mean of {len(SEEDS)} seeds)"); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3)
axes[0].set_ylabel("MSE"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR, "03_loss_curves.png"), dpi=120); plt.show()

In [ ]:
# Key metrics side by side (bar = mean, line = std over seeds, dashed = baseline)
show = ["Dyn MAE", "Dyn R2", "Dyn CCC", "Tempo MAE", "Tempo R2", "Tempo direction acc"]
model_names = list(MODELS)
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), show):
    ax.bar(range(len(model_names)), [mean.loc[n, col] for n in model_names], yerr=[std.loc[n, col] for n in model_names],
           color=[colors[n] for n in model_names], width=0.7, capsize=3, error_kw={"lw": 1, "ecolor": "#52514e"})
    ax.axhline(mean.loc["Baseline (mean)", col], color="#52514e", ls="--", lw=1, label="Baseline")
    ax.set_title(f"{col}  ({'lower' if BETTER[col] == 'low' else 'higher'} is better)", fontsize=10)
    ax.set_xticks([]); ax.grid(axis="y", alpha=0.3); ax.spines[["top", "right"]].set_visible(False)
handles = [plt.Rectangle((0, 0), 1, 1, color=colors[n]) for n in model_names] + [plt.Line2D([], [], color="#52514e", ls="--")]
fig.legend(handles, model_names + ["Baseline (mean)"], loc="lower center", ncol=6, fontsize=9, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.savefig(os.path.join(PLOTS_DIR, "04_metric_comparison.png"), dpi=120); plt.show()

In [ ]:
# Predicted vs ground truth for the first minute of one test piece (seed 42 models)
idx = np.where(piece_id[split["test"]] == test_p[0])[0][:12]
t_axis = np.arange(len(idx) * FRAMES) * HOP / SR
truth = Yte[idx].reshape(-1, 2).numpy()
fig, axes = plt.subplots(len(MODELS), 2, figsize=(15, 2.2 * len(MODELS)), sharex=True)
for row, n in zip(axes, MODELS):
    p = preds[n][idx].reshape(-1, 2).numpy()
    for ax, k, label, scale in ((row[0], 0, "Dynamics (velocity)", 127), (row[1], 1, "Tempo (log2 rate)", 1)):
        ax.plot(t_axis, truth[:, k] * scale, color="#52514e", lw=1, label="Ground truth")
        ax.plot(t_axis, p[:, k] * scale, color=colors[n], lw=2, label=n)
        ax.set_ylabel(label, fontsize=8); ax.grid(alpha=0.3); ax.legend(loc="upper right", fontsize=8)
axes[0, 0].set_title("Dynamics")
axes[0, 1].set_title("Tempo")
axes[-1, 0].set_xlabel("Time (s)"); axes[-1, 1].set_xlabel("Time (s)")
plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR, "05_predicted_vs_truth.png"), dpi=120); plt.show()